# Explore Role Clusters

Reads the outputs of `src/build_features.py` and `src/build_clusters.py` (run those first) and visually sanity-checks the per-role playstyle clusters. This notebook is exploration only -- the scripts in `src/` are the source of truth for the pipeline.

In [ ]:
import sys
sys.path.append('../src')

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from feature_config import ALL_FEATURE_FIELDS, VALID_ROLES
from playstyle_space import to_playstyle_space

FEATURE_COLUMNS = [f.name for f in ALL_FEATURE_FIELDS]

clusters_df = pd.read_parquet('../data/processed/participant_clusters.parquet')
clusters_df['teamPosition'].value_counts()

## PCA scatter per role

Reuses each role's fitted outcome adjuster + scaler + PCA from `models/` so the 2D projection matches what KMeans actually clustered on (first two PCA components).

In [ ]:
fig, axes = plt.subplots(1, len(VALID_ROLES), figsize=(24, 5))

for ax, role in zip(axes, VALID_ROLES):
    artifact = joblib.load(f'../models/{role.lower()}_cluster.joblib')
    role_df = clusters_df[clusters_df['teamPosition'] == role]

    reduced = to_playstyle_space(role_df, artifact)

    scatter = ax.scatter(reduced[:, 0], reduced[:, 1], c=role_df['cluster'], cmap='tab10', s=6, alpha=0.5)
    ax.set_title(f'{role} (k={artifact["chosen_k"]})')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

fig.tight_layout()
plt.show()

## Centroid comparison for a role

Bar chart of a few headline features per cluster, for one role at a time. Change `ROLE` below.

In [ ]:
ROLE = 'JUNGLE'
HEADLINE_FEATURES = [
    'damage_per_minute', 'gold_per_minute', 'vision_score_per_minute',
    'kill_participation', 'kda', 'cs_per_min',
]

role_df = clusters_df[clusters_df['teamPosition'] == ROLE]
centroids = role_df.groupby('cluster')[HEADLINE_FEATURES].mean()
win_rate = role_df.groupby('cluster')['win'].mean()
sizes = role_df.groupby('cluster').size()

print('cluster sizes:\n', sizes)
print('\nwin rate by cluster:\n', win_rate)

centroids.plot(kind='bar', figsize=(10, 5), title=f'{ROLE} cluster centroids')
plt.tight_layout()
plt.show()

## Tier mix per cluster

Where each tier's players fall across clusters, for one role at a time.

In [ ]:
pd.crosstab(role_df['cluster'], role_df['tier'], normalize='columns').round(3)